# 03 Generate SOCAT Across-Time Weights, DII, and ID Intermediates

Purpose: generate SOCAT across-time provenance intermediates used by the across-time figure components.

Inputs:
- pCO2 LEAP reconstruction zarr
- SOCAT mask zarr

Outputs:
- `Finalweights_SOCAT90s_16k_seed13.txt`
- `Finalweights_SOCAT2000s_16k_seed13.txt`
- `Finalweights_SOCAT2010s_16k_seed13.txt`
- `Finalweights_SOCAT_2018_2019_16k_seed13.txt`
- `Finalweights_SOCAT202022_16k_seed13.txt`
- `Finalimbs_SOCAT90s_16k_seed13.txt`
- `Finalimbs_SOCAT2000s_16k_seed13.txt`
- `Finalimbs_SOCAT2010s_16k_seed13.txt`
- `Finalimbs_SOCAT_2018_2019_16k_seed13.txt`
- `Finalimbs_SOCAT202022_16k_seed13.txt`
- `time_dim_all.csv`
- `time_scales_all.csv`

Estimated runtime: slow. Each time window runs DADAPy feature-weight optimization and intrinsic-dimension calculations.

Notes:
- This notebook is provenance material. This optional regeneration step requires the raw data and a compatible Python environment.
- It generates the across-time SOCAT feature-weight, DII, and intrinsic-dimension files used by the plotting scripts.
- The 2020-2022 intrinsic-dimension values are computed directly here so the across-time CSV files are self-contained.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from dadapy import Data
from dadapy.feature_weighting import FeatureWeighting
from sklearn.preprocessing import StandardScaler

# Local paths; replace with gs:// paths if running against cloud storage.
RECONSTRUCTION_ZARR = Path("../raw/pCO2_LEAP_fco2-residual-full-dataset-preML_198201-202412.zarr")
SOCAT_MASK_ZARR = Path("../raw/socat_mask_feb1982-dec2022.zarr")
OUTPUT_DIR = Path("../intermediates")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLE = 16000
RANDOM_STATE = 13
N_EPOCHS = 80
MAXK_CAP = 10000
RANGE_MAX = 1024


## Feature Definitions


In [ ]:
FEATURES = [
    "sst",
    "sst_anomaly",
    "sss",
    "sss_anomaly",
    "chl_log",
    "chl_log_anomaly",
    "mld_log",
    "xco2_trend",
    "A",
    "B",
    "C",
    "T0",
    "T1",
]
TARGET = "delta_fco2_1D"


## Time Windows

The labels match the filenames used by the plotting scripts where possible. `SOCAT202022` is included so `time_dim_all.csv` and `time_scales_all.csv` are self-contained.


In [ ]:
TIME_WINDOWS = {
    "SOCAT90s": ("1990-01-01", "1996-12-31", "1990-1996"),
    "SOCAT2000s": ("2001-01-01", "2004-12-31", "2001-2004"),
    "SOCAT2010s": ("2011-01-01", "2012-12-31", "2011-2012"),
    "SOCAT_2018_2019": ("2018-01-01", "2019-12-31", "2018-2019"),
    "SOCAT202022": ("2020-01-01", "2022-12-31", "2020-2022"),
}


## Helper Functions


In [ ]:
def load_socat_dataset(reconstruction_zarr, socat_mask_zarr):
    socat_mask = xr.open_dataset(socat_mask_zarr, engine="zarr")
    ds = xr.open_dataset(reconstruction_zarr, engine="zarr")
    ds = ds.sel(time=slice("1982-02-01", "2022-12-31"))

    aligned_mask = socat_mask.reindex(time=ds.time, method="nearest")
    ds = ds.where(aligned_mask.socat_mask == 1)
    ds[TARGET] = ds["fco2"] - ds["xco2_trend"]
    return ds[FEATURES + [TARGET]]

def sample_and_scale(frame, n_sample=N_SAMPLE, random_state=RANDOM_STATE):
    sample = frame.sample(n=n_sample, random_state=random_state)
    scaled = StandardScaler().fit_transform(sample.loc[:, FEATURES + [TARGET]])
    return pd.DataFrame(scaled, columns=FEATURES + [TARGET])

def compute_id_curve(scaled_frame):
    data = Data(scaled_frame.to_numpy(), verbose=False)
    data.compute_distances(maxk=min(scaled_frame.shape[0] - 1, MAXK_CAP))
    ids, _, scales = data.return_id_scaling_gride(range_max=RANGE_MAX)
    return ids, scales

def run_weight_dii_job(scaled_frame, output_tag, n_epochs=N_EPOCHS):
    print(f"Generating weights/DII for {output_tag}")
    target = FeatureWeighting(scaled_frame[[TARGET]].to_numpy(), verbose=True)
    inputs = FeatureWeighting(scaled_frame[FEATURES].to_numpy(), verbose=True)

    final_imbs, final_weights = inputs.return_backward_greedy_dii_elimination(
        target_data=target,
        initial_weights=None,
        n_epochs=n_epochs,
        learning_rate=None,
        decaying_lr="cos",
    )

    np.savetxt(OUTPUT_DIR / f"Finalweights_{output_tag}_16k_seed13.txt", final_weights)
    np.savetxt(OUTPUT_DIR / f"Finalimbs_{output_tag}_16k_seed13.txt", final_imbs)


## Load SOCAT-Masked Dataset


In [ ]:
DS_SOCAT = load_socat_dataset(RECONSTRUCTION_ZARR, SOCAT_MASK_ZARR)
DS_SOCAT


## Generate Across-Time Outputs

Slow cell. For each window, this computes:
- feature weights and DII curve text files,
- one intrinsic-dimension curve for all features plus target.


In [ ]:
time_dim_all = pd.DataFrame()
time_scales_all = pd.DataFrame()

for output_tag, (start, stop, label) in TIME_WINDOWS.items():
    print(f"Window {label}: {start} to {stop}")
    frame = DS_SOCAT.sel(time=slice(start, stop)).to_dataframe().dropna()
    print("Available rows:", frame.shape[0])

    scaled = sample_and_scale(frame)
    run_weight_dii_job(scaled, output_tag)

    ids, scales = compute_id_curve(scaled)
    time_dim_all[output_tag] = ids
    time_scales_all[output_tag] = scales

# Plotting script expects this shorter column name for the 2018-2019 window.
time_dim_all = time_dim_all.rename(columns={"SOCAT_2018_2019": "SOCAT201819"})
time_scales_all = time_scales_all.rename(columns={"SOCAT_2018_2019": "SOCAT201819"})

time_dim_all.to_csv(OUTPUT_DIR / "time_dim_all.csv", index=False)
time_scales_all.to_csv(OUTPUT_DIR / "time_scales_all.csv", index=False)


## Outputs Created

Quick check:


In [ ]:
expected_outputs = [
    "Finalweights_SOCAT90s_16k_seed13.txt",
    "Finalweights_SOCAT2000s_16k_seed13.txt",
    "Finalweights_SOCAT2010s_16k_seed13.txt",
    "Finalweights_SOCAT_2018_2019_16k_seed13.txt",
    "Finalweights_SOCAT202022_16k_seed13.txt",
    "Finalimbs_SOCAT90s_16k_seed13.txt",
    "Finalimbs_SOCAT2000s_16k_seed13.txt",
    "Finalimbs_SOCAT2010s_16k_seed13.txt",
    "Finalimbs_SOCAT_2018_2019_16k_seed13.txt",
    "Finalimbs_SOCAT202022_16k_seed13.txt",
    "time_dim_all.csv",
    "time_scales_all.csv",
]

for filename in expected_outputs:
    path = OUTPUT_DIR / filename
    print(filename, "OK" if path.exists() else "MISSING")
